# 21.17. MP-05 — статистическая модель формы сердца

**Статус: выполнен локальный пилот; результат математической модели, не анатомическая и не FEM-валидация.** Проверяемый вопрос: насколько маломерный базис, обученный на других людях, воспроизводит форму известной опорной маски человека, исключённого из обучения? Этот расчёт не предсказывает движение или невидимую фазу. Исходные КТ и маски не входят в ноутбук.


## Ник как основной проверочный человек

В существующем тесте leave-one-person-out Ник уже полностью исключён из построения базиса. Для него Dice средней формы `0,8095`, с одной PCA-компонентой `0,8236`. Это **проекция с использованием его опорной автоматической маски**, не прогноз формы без КТ и не проверка независимой анатомии. Adam и Georg оставлены как обучающие/сравнительные люди; исследовательская оценка одной компоненты на трёх людях не позволяет объявить модель пригодной для пациентов.


## Входы, разделение и QC

**Факт о данных:** три разных псевдонима — Adam, Georg, Nix. Использована только автоматическая маска `whole_heart` фазы `phase_00` из `products_v2` для каждого человека. Источник — TotalSegmentator v2; манифест масок имеет статус `automatic_products_pending_manual_review`. Независимого принятого ручного референса здесь нет. Фактические тома КТ и маски находятся во внешнем локальном хранилище и передаются скрипту только аргументом.

**Допущение модели:** маска каждого человека центрируется по центру её объёма и дискретизируется на общей сетке 64³ с шагом 3,5 мм; подписанное расстояние до границы (мм) образует вектор формы. Такая нормализация убирает абсолютное положение и снижает разрешение, поэтому результат нельзя напрямую переносить на индивидуальную сетку FEM. Для каждого исключённого человека PCA обучается на двух остальных. Его собственная `phase_00` используется только для проекции и оценки сжатия; все фазы `01+` исключены полностью.

**QC:** исходная маска должна быть непустой, бинарной и полностью помещаться на нормализованную сетку. В выходе скрипта есть SHA-256 каждой опорной маски. Разбиение по псевдонимам проверяется программно. Проверка тождества людей за пределами псевдонимов и независимая анатомическая точность масок остаются открытыми.


## Реальный расчёт 28.09.2026

| Исключённый человек | Dice средней формы, 0 компонент | Dice, 1 компонента | Изменение Dice |
|---|---:|---:|---:|
| Adam | 0,8625 | 0,8532 | −0,0094 |
| Georg | 0,7839 | 0,8250 | +0,0411 |
| Nix | 0,8095 | 0,8236 | +0,0141 |

**Отрицательный результат:** одна компонента ухудшила Dice у Adam. Два обучающих человека дают максимум одну содержательную компоненту в каждом разбиении. Эти результаты не подтверждают ни оптимальную размерность, ни перенос на популяцию. Проекция использует тестовую опорную маску, поэтому её результат нельзя выдавать за предсказание отсутствующей КТ. Согласованных вершин между людьми для MeshGraphNet или FEM этим методом не получено.


## Продолжение: PCA workflow MONAI Physio на синтетике

**Внутренний вычислительный тест:** в установленном MONAI Physio `2026.09.1` вызван `WorkflowCreateStatisticalModel.process()` для пяти обучающих и одной исключённой синтетической поверхности (362 вершины каждая). Две гладкие моды заданы при копировании одного шаблона; совпадение индексов вершин и граней проверено. Подкласс использует известное соответствие и пропускает ICP/deformable registration; настоящие этапы выделения поверхности, PCA и результата MONAI выполняются. Это проверка PCA API, не проверка поиска соответствий на КТ.

| Число компонент | RMSE вершин исключённой поверхности, мм |
|---:|---:|
| 0 | 0,231853 |
| 1 | 0,221267 |
| 2 | 2,11·10⁻¹⁴ |

Объяснённая дисперсия: 0,664393 и 0,335607. Почти точный результат с двумя компонентами ожидаем для специально созданной двухмодовой геометрии и не означает точности на людях. Исполняемый [mp05_07_monai_pca_smoke.py](mp05_07_monai_pca_smoke.py) сохраняет синтетические `summary.json` и VTP только в игнорируемую Git папку `runs/mp05_07_synthetic_pca/`.


## Воспроизведение и граница вывода

Исполняемый расчёт: [mp05_07_eval.py](mp05_07_eval.py), команда `python mp05_07_eval.py pca --data-root <локальный_корень_производных_4D_КТ>` в среде с `numpy`, `scipy`, `nibabel`. Подробности входов, среды и соседних задач: [mp05_07_results.md](mp05_07_results.md). Скрипт ничего не записывает; его stdout содержит агрегаты и контрольные суммы, локальные абсолютные пути не сохраняются.

**Нерешённый вопрос:** оценить устойчивость базиса на большем независимом корпусе с проверенным происхождением масок, затем влияние геометрической потери на `Z` и `ΔZ` при неизменных материалах, электродах и FEM-постановке.
